# 04 · Regularization and How to Choose It

*Stabilize distributed slip and make the trade-off defensible.*

**Learning objectives**

- derive the regularized estimator and augmented system
- state the assumptions encoded by smoothing, damping, and stress penalties
- interpret regularization through SVD filter factors and Gaussian priors
- compare L-curve, ABIC, and cross-validation choices

**Prerequisites:** Chapter 03; weighted least squares and the SVD  
**Estimated time:** 90–120 minutes

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

Chapter 03 showed that data fit alone does not select a stable slip model.
Regularization supplies missing information explicitly. The scientific task is
therefore twofold: choose what property of the model to prefer, then choose how
strongly to prefer it. Neither choice is assumption-free.


## Movement 1 — regularization as filtering and prior information

Regularization minimizes

$$\Phi=(Gm-d)^TW(Gm-d)+\lambda\lVert L(m-m_{ref})\rVert^2.$$

It is equivalent to appending $\sqrt{\lambda}L$ to the whitened design matrix
and $\sqrt{\lambda}Lm_{ref}$ to the data. For damping, each SVD mode is
multiplied by $\sigma_i^2/(\sigma_i^2+\lambda)$: poorly observed modes are
suppressed first. A Laplacian instead penalizes curvature, schematically
$m_{i-1}-2m_i+m_{i+1}$ in one dimension. Stress-kernel regularization makes a
different, elastic-interaction assumption.

The same expression is a Gaussian-prior MAP estimate: $L$ defines prior
directions and $\lambda$ their precision. Regularization trades variance for
bias; it never restores wavelengths absent from the data.

## Movement 2 — choosing $\lambda$

The L-curve balances misfit and penalty norm geometrically. ABIC integrates the
linear slip parameters out and compares marginal likelihood, including an
Occam factor for model volume. Cross-validation asks which strength predicts
held-out observations. Spatially neighboring folds can leak correlated
information, so ordinary random-fold CV can be optimistic for InSAR.


## Tikhonov regularization

We add a penalty term to the misfit that punishes "bad" models:

$$ \min_{\mathbf{m}} \;
\underbrace{\lVert G\mathbf{m} - \mathbf{d}\rVert^2_{C_d}}_{\text{fit the data}}
\;+\;
\underbrace{\lambda\,\lVert L(\mathbf{m} - \mathbf{m}_{\text{ref}})\rVert^2}_{\text{stay simple}}. $$

- $L$ is the **regularization operator**: it measures how "bad" a model is.
- $\lambda$ is the **regularization strength**: how much we trust the prior
  versus the data. $\lambda \to 0$ recovers the unregularized solution of
  notebook 03; $\lambda \to \infty$ forces $\mathbf{m} \to \mathbf{m}_{\text{ref}}$.
- $\mathbf{m}_{\text{ref}}$ is a **reference model** (default $\mathbf 0$).

### Choices of $L$
- **Smoothing** (`'laplacian'`): $L$ is the discrete Laplacian, which measures
  slip *roughness*. Penalizing it favors spatially smooth slip.
- **Damping** (`'damping'`): $L = I$, which measures slip *magnitude*. This is
  the minimum-norm / minimum-moment prior.
- **Stress kernel** (`'stresskernel'`): $L$ is built from inter-patch elastic
  stress interactions — a physically motivated smoothness.

### The augmented-system view
Regularization is just **extra equations**. Stacking

$$ \begin{bmatrix} G \\ \lambda L \end{bmatrix} \mathbf{m}
   = \begin{bmatrix} \mathbf{d} \\ \lambda L\,\mathbf{m}_{\text{ref}} \end{bmatrix} $$

and solving by ordinary least squares gives exactly the regularized solution.
The prior enters as synthetic "data" with weight $\lambda$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.data.gnss(lon=glon, lat=glat, east=_zero, north=_zero, up=_zero, sigma_east=_one, sigma_north=_one, sigma_up=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.data.gnss(
    lon=glon, lat=glat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_sta, sigma), sigma_north=np.full(n_sta, sigma), sigma_up=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

## One smoothed inversion

Add `regularization='laplacian'` and a strength. Compare to the true slip and to the
unregularized catastrophe from notebook 03.

In [ ]:
raw = geodef.solve(fault, gnss, components="dip")                 # lambda = 0
sm = geodef.solve(fault, gnss, components="dip",
                   regularization="laplacian", regularization_strength=1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = slip_true[N:].max()
lim_raw = abs(raw.dip_slip).max()
geodef.plot.slip(fault, slip_true[N:], ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True", colorbar_label="Slip (m)")
geodef.plot.slip(fault, raw.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-lim_raw, vmax=lim_raw, title="Unregularized",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, sm.dip_slip, ax=axes[2], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Laplacian, lambda=1",
                 colorbar_label="Slip (m)")
plt.tight_layout()

Smoothing recovers the bump. It does not reproduce the true slip exactly — some
detail is unresolvable from noisy surface data — but it is smooth, positive
where it should be, and physically plausible.

## The effect of $\lambda$

Sweep the strength from tiny to huge. Small $\lambda$ under-smooths (noisy);
large $\lambda$ over-smooths (washed out, and the moment is suppressed). The
"right" $\lambda$ balances the two — notebook 05 makes that choice quantitative.

The top row shows the recovered slip on a single shared diverging scale; the bottom row shows each solution's error (recovered $-$ true). Under-smoothing leaves noisy, sign-alternating residuals while over-smoothing leaves a broad, one-sided deficit.

In [ ]:
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0]
truth = slip_true[N:]
vmax = truth.max()
ncol = len(lambdas) + 1
fig, axes = plt.subplots(2, ncol, figsize=(2.6 * ncol, 5.6),
                         constrained_layout=True)

# Row 0: true + recovered slip, one shared symmetric RdBu_r scale.
geodef.plot.slip(fault, truth, ax=axes[0, 0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
recovered = []
for ax, lam in zip(axes[0, 1:], lambdas):
    r = geodef.solve(fault, gnss, components="dip",
                      regularization="laplacian", regularization_strength=lam)
    recovered.append(r.dip_slip)
    geodef.plot.slip(fault, r.dip_slip, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False,
                     title=f"lambda={lam:g}\nchi2={r.reduced_chi2:.2f}")

# Row 1: recovered - true (error), on their own shared symmetric scale.
diffs = [m - truth for m in recovered]
dmax = max(np.abs(d).max() for d in diffs)
axes[1, 0].axis("off")
for ax, lam, d in zip(axes[1, 1:], lambdas, diffs):
    geodef.plot.slip(fault, d, ax=ax, cmap="RdBu_r",
                     vmin=-dmax, vmax=dmax, colorbar=False,
                     title=f"error (lambda={lam:g})")

fig.colorbar(axes[0, 0].collections[-1], ax=axes[0, :], shrink=0.8,
             label="Slip (m)")
fig.colorbar(axes[1, 1].collections[-1], ax=axes[1, :], shrink=0.8,
             label="Recovered - true (m)")

## Damping vs. smoothing

Different priors encode different beliefs. Damping (`L = I`) pulls slip toward
*zero* everywhere (minimum norm), which tends to shrink the amplitude; smoothing
penalizes *roughness* but is happy with a large smooth bump. Here they are at a
comparable strength.

In [ ]:
sm = geodef.solve(fault, gnss, components="dip",
                   regularization="laplacian", regularization_strength=1.0)
dm = geodef.solve(fault, gnss, components="dip",
                   regularization="damping", regularization_strength=1.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, sm.dip_slip, ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Smoothing (Laplacian)",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, dm.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Damping (minimum norm)",
                 colorbar_label="Slip (m)")
plt.tight_layout()
print(f"smoothing moment magnitude: Mw {sm.Mw:.2f}")
print(f"damping   moment magnitude: Mw {dm.Mw:.2f}")

## Movement 2 worked demonstrations: choosing the strength


## The L-curve

Every $\lambda$ produces one (misfit, roughness) pair,

$$ \rho(\lambda) = \lVert G\hat{\mathbf m}_\lambda - \mathbf d\rVert,
   \qquad
   \eta(\lambda) = \lVert L\hat{\mathbf m}_\lambda\rVert . $$

Plotted on log-log axes as $\lambda$ sweeps, these trace an **L-shaped** curve.
The steep branch (small $\lambda$) is the *over-fitting* regime, where roughness
explodes while the misfit barely improves; the flat branch (large $\lambda$) is
the *over-smoothing* regime, where the misfit grows for no roughness gain. The
**corner** joining them is the best compromise, and `geodef` locates it at the
point of **maximum curvature**

$$ \lambda^\star = \arg\max_\lambda\;
   \kappa(\lambda) = \frac{\rho'\,\eta'' - \rho''\,\eta'}
   {\big((\rho')^2 + (\eta')^2\big)^{3/2}}, $$

with the derivatives taken along the log-log curve. `geodef.invert.lcurve()` sweeps
$\lambda$, returns the $\rho$ and $\eta$ arrays plus the located corner, and
`.plot()` marks it.

In [ ]:
lc = geodef.invert.lcurve(fault, gnss, regularization="laplacian", components="dip",
                   regularization_range=(1e-2, 1e3), n=40)
lc.plot()
print(f"L-curve corner: lambda = {lc.optimal:.3g}")

## ABIC

The **Akaike Bayesian Information Criterion** treats the roughness penalty as a
Gaussian *prior* on the slip and the smoothing strength $\lambda$ as a
hyperparameter setting the prior variance. Integrating (marginalizing) the slip
$\mathbf m$ out of the posterior leaves a marginal likelihood for $\lambda$
alone; ABIC is $-2\log$ of it, so the **minimum** is the most probable strength.
For the linear-Gaussian problem (Fukuda & Johnson, 2008),

$$ \mathrm{ABIC}(\lambda) = N\,\log\!\Big(
   \lVert G\mathbf m_\lambda - \mathbf d\rVert_W^2
   + \lambda\,\lVert L\mathbf m_\lambda\rVert^2 \Big)
   \;-\; \log\big|\lambda\,L^{\mathsf T}L\big|_{+}
   \;+\; \log\big|G^{\mathsf T}WG + \lambda\,L^{\mathsf T}L\big|, $$

where $N$ is the number of data, $\lVert\mathbf r\rVert_W^2 =
\mathbf r^{\mathsf T} W\mathbf r$ is the noise-weighted misfit, and $|\cdot|_+$
is the product of the *positive* eigenvalues (a pseudo-determinant, since
$L^{\mathsf T}L$ is rank-deficient for a Laplacian). The first term rewards
fitting the data; the two log-determinants form an **Occam factor** that
penalizes an over-flexible prior, so ABIC trades data fit against model
complexity automatically. `geodef.invert.abic_curve()` evaluates it across the sweep
(and `regularization_strength='abic'` selects the minimizer directly).

In [ ]:
ab = geodef.invert.abic_curve(fault, gnss, regularization="laplacian", components="dip",
                       regularization_range=(1e-2, 1e3), n=40)
ab.plot()
print(f"ABIC minimum:   lambda = {ab.optimal:.3g}")

## Cross-validation

$k$-fold **cross-validation** picks the $\lambda$ that best predicts data it
never saw. Partition the observations into $k$ disjoint folds. For each fold
$f$, fit the model on the *other* $k-1$ folds at strength $\lambda$ to get
$\hat{\mathbf m}^{(-f)}_\lambda$, then predict the held-out fold and accumulate
its squared error:

$$ \mathrm{CV}(\lambda) = \sum_{f=1}^{k}
   \big\lVert \mathbf d_f - G_f\,\hat{\mathbf m}^{(-f)}_\lambda \big\rVert^2 . $$

The $\lambda$ minimizing this out-of-sample error is chosen. Unlike the L-curve
and ABIC it needs no explicit prior or curvature heuristic -- only the data's
own predictive performance -- at the cost of $k$ inversions per trial
$\lambda$. Pass `regularization_strength='cv'` (with `cv_folds=k`) and `invert()`
runs the whole procedure internally, choosing the $\lambda$ that best predicts
held-out stations.

In [ ]:
cv = geodef.solve(fault, gnss, components="dip",
                   regularization="laplacian", regularization_strength="cv", cv_folds=5)
print(f"CV-selected:    lambda = {cv.regularization_strength:.3g}")

## Do they agree?

The three criteria are derived from different principles, so they need not land
on the same number — but they usually agree to within a factor of a few. Here are
all three choices and the slip each produces.

The top row is the slip each criterion produces (shared symmetric scale); the bottom row is each one's difference from the truth (recovered $-$ true). The residual maps are nearly identical, confirming the criteria land on very similar models.

In [ ]:
choices = {
    "L-curve": lc.optimal,
    "ABIC": ab.optimal,
    "cross-val": cv.regularization_strength,
}
truth = slip_true[N:]
vmax = truth.max()
ncol = len(choices) + 1
fig, axes = plt.subplots(2, ncol, figsize=(3 * ncol, 6),
                         constrained_layout=True)

# Row 0: true + each method's slip, shared symmetric RdBu_r scale.
geodef.plot.slip(fault, truth, ax=axes[0, 0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
recovered = {}
for ax, (name, lam) in zip(axes[0, 1:], choices.items()):
    r = geodef.solve(fault, gnss, components="dip",
                      regularization="laplacian", regularization_strength=lam)
    recovered[name] = r.dip_slip
    geodef.plot.slip(fault, r.dip_slip, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False,
                     title=f"{name}\nlambda={lam:.2g}")

# Row 1: method - true, so the (small) differences between methods show.
diffs = {name: m - truth for name, m in recovered.items()}
dmax = max(np.abs(d).max() for d in diffs.values())
axes[1, 0].axis("off")
for ax, (name, d) in zip(axes[1, 1:], diffs.items()):
    geodef.plot.slip(fault, d, ax=ax, cmap="RdBu_r",
                     vmin=-dmax, vmax=dmax, colorbar=False,
                     title=f"{name} - true")

fig.colorbar(axes[0, 0].collections[-1], ax=axes[0, :], shrink=0.8,
             label="Slip (m)")
fig.colorbar(axes[1, 1].collections[-1], ax=axes[1, :], shrink=0.8,
             label="Recovered - true (m)")

## Checkpoint questions

**What does $L$ specify separately from $\lambda$?**

<details><summary>Answer</summary>



The directions and structure being penalized; lambda specifies strength.



</details>

**Why can L-curve, ABIC, and CV disagree?**

<details><summary>Answer</summary>



They optimize different criteria under different assumptions.



</details>

**Does larger lambda always mean smoother models across operators?**

<details><summary>Answer</summary>



No. Operator scaling and null spaces differ.



</details>


## Common mistakes

- Writing $\lambda^2$ when the API already uses lambda changes the intended strength.
- Calling damping 'smoothing' hides its minimum-norm assumption.
- Selecting lambda after looking only at the preferred slip map is circular.


## Recap

- Regularization filters poorly observed modes and trades variance for bias.
- The operator is a scientific prior; strength controls its influence.
- L-curve, ABIC, and CV answer related but distinct selection questions.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/04_regularization_solutions.ipynb`.

1. Match Laplacian and damping solutions by misfit, then compare structure.
2. Use a nonzero reference model and identify which patches follow it.
3. Compare L-curve, ABIC, and CV strengths after doubling noise.
4. Challenge: construct a first-difference operator and select its strength with ABIC.


## Further reading

- Tikhonov & Arsenin (1977), regularization theory.
- Hansen (1992), analysis of the L-curve.
- Akaike (1980), likelihood and Bayesian model selection.
- Fukuda & Johnson (2008), geodetic Bayesian regularization.
